In [ ]:
#Download the dependencis
#!pip install xgboost

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import random

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score
)
from xgboost import XGBClassifier

# Seed
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# Load data
df = pd.read_csv('/containt/ML-Martensite Morphology-data.csv')
df.columns = df.columns.str.strip()

# Features and target
X = df.drop(columns=["Martensite Morphology"])
y = df["Martensite Morphology"]

le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Train/test split (80:20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.20, random_state=SEED
)

# Pipeline of the models
pipelines = {
    "Logistic Regression": Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(
            max_iter=500,
            random_state=SEED,
            penalty='l2',
            solver='lbfgs'
        ))
    ]),
    "Random Forest": Pipeline([
        ('clf', RandomForestClassifier(random_state=SEED))
    ]),
    "XGBoost": Pipeline([
        ('clf', XGBClassifier(
            eval_metric='mlogloss',
            random_state=SEED
        ))
    ])
}

# Grid parameters
param_grids = {
    "Logistic Regression": {
        'clf__C': [0.01, 0.1, 1, 10]
    },
    "Random Forest": {
        'clf__n_estimators': [50, 100, 200, 300],
        'clf__max_depth': [5, 10, 20, None],
        'clf__min_samples_leaf': [1, 2, 4, 5]
    },
    "XGBoost": {
        'clf__n_estimators': [50, 100, 200, 300],
        'clf__max_depth': [3, 5, 10, 20],
        'clf__learning_rate': [0.05, 0.1],
        'clf__reg_alpha': [0, 0.05, 0.1, 0.5, 1],
        'clf__reg_lambda': [1, 5, 10, 15]
    }
}

# Model training
results = {}
for name in pipelines:
    print(f"Training {name}...")
    grid = GridSearchCV(pipelines[name], param_grids[name], cv=5, scoring='accuracy')
    grid.fit(X_train, y_train)

    results[name] = {
        "model": grid.best_estimator_,
        "params": grid.best_params_,
        "accuracy": grid.score(X_test, y_test),
        "y_pred": grid.predict(X_test),
        "train_score": grid.best_estimator_.score(X_train, y_train)
    }

# Accuracy comparison plot
def plot_accuracy_comparison(results_dict):
    accs = [results_dict[m]['accuracy'] for m in results_dict]
    names = list(results_dict.keys())

    plt.figure(figsize=(7, 4))
    sns.barplot(x=accs, y=names, palette='crest')
    plt.xlabel('Accuracy')
    plt.title('Model Comparison - Test Accuracy')
    plt.xlim(0, 1)

    for i, acc in enumerate(accs):
        plt.text(acc + 0.01, i, f"{acc:.2f}", va='center')

    plt.tight_layout()
    plt.show()

plot_accuracy_comparison(results)

# Confusion matrix plot
def plot_confusion_matrix(y_true, y_pred, labels, title):
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm_norm, annot=cm, fmt='d', cmap="Blues",
        xticklabels=labels, yticklabels=labels
    )
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.show()

for name in results:
    plot_confusion_matrix(y_test, results[name]["y_pred"], le.classes_, f"{name} - Confusion Matrix")

# Train vs Test plot
def plot_train_vs_test(results_dict):
    names = list(results_dict.keys())
    train_accs = [results_dict[m]['train_score'] for m in names]
    test_accs = [results_dict[m]['accuracy'] for m in names]

    x = np.arange(len(names))
    width = 0.35

    fig, ax = plt.subplots(figsize=(8, 5))
    bars1 = ax.bar(x - width/2, train_accs, width, label='Train Accuracy', color='skyblue')
    bars2 = ax.bar(x + width/2, test_accs, width, label='Test Accuracy', color='salmon')

    for bar in bars1:
        height = bar.get_height()
        ax.annotate(
            f'{height:.2f}',
            xy=(bar.get_x() + bar.get_width() / 2, height),
            xytext=(0, 4),
            textcoords="offset points",
            ha='center',
            va='bottom'
        )

    for bar in bars2:
        height = bar.get_height()
        ax.annotate(
            f'{height:.2f}',
            xy=(bar.get_x() + bar.get_width() / 2, height),
            xytext=(0, 4),
            textcoords="offset points",
            ha='center',
            va='bottom'
        )

    ax.set_xticks(x)
    ax.set_xticklabels(names)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel('Accuracy')
    ax.set_title('Train vs Test Accuracy')
    ax.legend(loc='lower left')
    plt.tight_layout()
    plt.show()

plot_train_vs_test(results)

# Feature importance map plot
def get_feature_importance(model, model_name, feature_names):
    if model_name == "Logistic Regression":
        coefs = model.named_steps['clf'].coef_
        importances = np.mean(np.abs(coefs), axis=0)
    else:
        importances = model.named_steps['clf'].feature_importances_

    return pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances,
        'Model': model_name
    })

df_imp_all = pd.concat([
    get_feature_importance(results[m]["model"], m, X.columns) for m in results
])

df_imp_all['Normalized Importance'] = df_imp_all.groupby('Model')['Importance'].transform(
    lambda x: x / x.max()
)


wrap_map = {
    'Ferrite Grain size (micron)': 'Ferrite Grain\nsize (µm)',
    'Martensite Phase %': 'Martensite\nPhase %',
    'Ferrite Phase %': 'Ferrite\nPhase %',
    'Amount of Deformation% at IA temperature': 'Deformation %\nat IA Temp',
    'IA Temperature (°C)': 'IA Temp\n(°C)',
    'IA Time (second)': 'IA Time\n(sec)'
}
df_imp_all['Feature'] = df_imp_all['Feature'].replace(wrap_map)

# Heatmap plot
heatmap_data = df_imp_all.pivot(index='Model', columns='Feature', values='Normalized Importance')
heatmap_data = heatmap_data[heatmap_data.mean().sort_values(ascending=False).index]

# Metrics dataframe
metrics_df = pd.DataFrame([
    {
        'Accuracy': results[m]['accuracy'],
        'Precision': precision_score(y_test, results[m]['y_pred'], average='weighted', zero_division=0),
        'Recall': recall_score(y_test, results[m]['y_pred'], average='weighted', zero_division=0),
        'F1': f1_score(y_test, results[m]['y_pred'], average='weighted', zero_division=0)
    }
    for m in results
], index=results.keys())

# Combined plot
fig, (ax1, ax2) = plt.subplots(
    1, 2, figsize=(16, 6), gridspec_kw={'width_ratios': [3, 1]}
)

sns.heatmap(
    heatmap_data, annot=True, fmt=".2f", cmap="RdYlGn", linewidths=0.5,
    cbar_kws={'label': 'Normalized Importance'}, ax=ax1
)
ax1.set_title('Feature Importance Across Models', fontsize=14, weight='bold')
ax1.set_xlabel('Feature', fontsize=12)
ax1.set_ylabel('Model', fontsize=8)
ax1.tick_params(axis='x', labelrotation=90)
ax1.tick_params(axis='y', labelrotation=0)

sns.heatmap(
    metrics_df, annot=True, fmt=".2f", cmap="Greens",
    cbar=False, linewidths=0.5, ax=ax2
)
ax2.set_title('Model Performance Metrics', fontsize=14, weight='bold')
ax2.set_xlabel('Metric', fontsize=12)
ax2.set_ylabel('Model', fontsize=12)
ax2.yaxis.set_label_position('left')
ax2.yaxis.set_ticks_position('left')
ax2.tick_params(axis='x', rotation=0)
ax2.tick_params(axis='y', labelrotation=0)

plt.tight_layout()
plt.show()